#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [175]:
import pandas as pd 
from importlib import reload
import random

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

import Algoritmos.greedy as greedy_module
reload(greedy_module)
from Algoritmos.greedy import solucion_greedy

import Algoritmos.relocate as relocate_module
reload(relocate_module)
from Algoritmos.relocate import relocate

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")

cajas_nuevas = pd.read_csv("4r.cajas_nuevas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_3mm.csv")

In [176]:
grosor = 3

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Almacenamos también las cajas asignables a cada producto en un diccionario.

In [177]:
def guardar_cajas_y_productos(grosor=grosor):
    
    cajas = {
        row["caja_tipo_id"]: Caja(
            caja_id=row["caja_tipo_id"],
            dim_interior_ancho=row["caja_interior_ancho"],
            dim_interior_largo=row["caja_interior_largo"],
            dim_interior_alto=row["caja_interior_alto"]
        )
        for _, row in cajas_nuevas.iterrows()
    }

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = {
        row["codigo_producto"]: Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    }
    
    cajas_asignables_por_producto = {}

    for codigo, group in factibilidad.groupby('codigo_producto'):
        # Obtener IDs de los tipos de cajas
        cajas_ids_unicos = list(group['caja_tipo_id'].unique())
        
        cajas_producto = []
        for caja_id in cajas_ids_unicos:
            cajas_producto.append(caja_id)
            
        cajas_asignables_por_producto[codigo] = cajas_producto
                
    # Elegir grosor
    for caja_id, caja in cajas.items():
        caja.elegir_grosor(grosor_mm=grosor)
        
    return cajas, productos, cajas_asignables_por_producto

In [178]:
def ordenar_por_cajas_asignables(asignaciones_por_producto):
    '''
    Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
    '''
    productos_conteo = []

    for codigo_producto, cajas_asignables in asignaciones_por_producto.items():
        cantidad_cajas = len(cajas_asignables)  # Número de cajas asignables para este producto    
        productos_conteo.append({
            'codigo_producto': codigo_producto,
            'cantidad_cajas_asignables': cantidad_cajas
        })

    # Ordenar de menor a mayor cantidad de cajas
    productos_ordenados = sorted(
        productos_conteo,
        key=lambda x: x['cantidad_cajas_asignables'],
        reverse=False
    )

    lista_productos_ordenados = [item['codigo_producto'] for item in productos_ordenados]
    return lista_productos_ordenados

In [179]:
cajas, productos, asignaciones_por_producto = guardar_cajas_y_productos()
lista_productos_ordenados = ordenar_por_cajas_asignables(asignaciones_por_producto)

solucion_greedy = solucion_greedy(cajas, productos, lista_productos_ordenados, asignaciones_por_producto,
                                  criterio_greedy="maximizar_utilizacion_pallet",
                                  titulo_solucion="Greedy maximizando utilizacion pallet | Ordenamiento según #cajas asignables",
                                  grosor=grosor)

In [180]:
relocate(solucion_greedy, productos, cajas, asignaciones_por_producto, 
         titulo_solucion="Greedy + Relocate maximizando utilizacion pallet")

Hola
189345165.3
Baja de  189345165.3 a  189343218.96
189345165.3
Baja de  189345165.3 a  189343770.66000003
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
Baja de  189345165.3 a  189340708.8
Baja de  189340708.8 a  189340153.68
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
Baja de  189345165.3 a  189342148.32
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
Baja de  189345165.3 a  189344840.28
189345165.3
Baja de  189345165.3 a  189335700.42000002
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
Baja de  189345165.3 a  189344373.96
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
189345165.3
Baja de  189345165.3 a  189345158.22
189345165.3
189345165.3
189345165.3
189

KeyboardInterrupt: 

In [ ]:
solucion_greedy.resumen_general()